# ⚡ Simple NVIDIA Nemotron-3-Nano-30B Inference via vLLM

This notebook provides a clean, single-prompt inference script for the **NVIDIA-Nemotron-3-Nano-30B** base model. It is directly inspired by the environment setup, package extraction, and vLLM configuration found in the competition baseline submissions.

## 🛠️ Step 1: Environment Setup & Package Unpacking

We uninstall the default PyTorch stack and unpack the custom Blackwell-optimized environment packages provided via Kaggle's metric utility script.

In [ ]:
import subprocess
import sys

commands = [
    "uv pip uninstall torch torchvision torchaudio",
    "tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell",
]

for cmd in commands:
    print(f"Running: {cmd}")
    subprocess.run(cmd, shell=True, check=True)

# Prepend the unpacked environment directory to python search path
sys.path.insert(0, "/tmp")

## 📥 Step 2: Configure Environment Variables & Load Model

In [ ]:
import os

# Set Triton and Transformers flags for offline execution
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"

from vllm import LLM, SamplingParams
import kagglehub

# Download or locate the base model path
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
print(f"Using model path: {MODEL_PATH}")

# Initialize vLLM Engine
llm = LLM(
    model=str(MODEL_PATH),
    tensor_parallel_size=1,
    max_num_seqs=64,
    gpu_memory_utilization=0.85,
    dtype="auto",
    max_model_len=8192,
    trust_remote_code=True,
    enable_lora=False,  # base model only
)

## 💬 Step 3: Define Inference Function & Run Prompt

In [ ]:
def generate_response(prompt_text: str, temperature: float = 0.0, max_tokens: int = 2048) -> str:
    """Formats the prompt using the chat template and generates a response from the vLLM engine."""
    sampling_params = SamplingParams(
        temperature=temperature,
        top_p=1.0,
        max_tokens=max_tokens,
    )
    tokenizer = llm.get_tokenizer()
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        tokenize=False,
        add_generation_prompt=True,
    )
    outputs = llm.generate([prompt], sampling_params=sampling_params)
    return outputs[0].outputs[0].text

# Example Usage:
prompt = "Convert the number 42 into Roman numerals."
print(f"Prompt: {prompt}\n")

print("Generating response...")
response = generate_response(prompt)

print("\n" + "="*40 + " RESPONSE " + "="*40)
print(response)
print("="*90)

## 📊 Step 4: Run Diagnostic Batch Inference on Dataset

We load the training dataset, prepare prompts using the custom system prompt and template, run fast batch generation using the vLLM engine, and output diagnostic statistics including accuracy and failure cases.

In [ ]:
import pandas as pd
import re

SYSTEM_PROMPT = """You are an expert pattern recognition solver competing in a reasoning challenge.

You will be given a puzzle from Alice's Wonderland where a secret transformation rule converts inputs to outputs.

Your approach MUST follow these steps inside <think> tags:
1. HYPOTHESIZE: Study each input-output example and state the rule clearly
2. VERIFY: Apply your rule to every example to confirm it holds
3. APPLY: Use the confirmed rule on the test input

Your final answer MUST be inside \\boxed{} and nothing after it.

Format:
<think>
HYPOTHESIZE: ...
VERIFY: ...
APPLY: ...
</think>
\\boxed{your_answer}"""

def build_prompt(puzzle_text: str) -> list[dict]:
    """
    puzzle_text is the raw 'prompt' column from train.csv / test.csv
    Returns messages list ready for vLLM or Claude API
    """
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": puzzle_text}
    ]

def extract_boxed_content(text):
    """
    Extracts the content inside \\boxed{...} LaTeX blocks.
    Prioritizes the last matching boxed structure, handling basic nested brackets.
    """
    matches = re.findall(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}", text)
    if matches:
        return matches[-1].strip()
    
    # Fallback heuristics
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    if lines:
        last_line = lines[-1]
        numbers = re.findall(r"[-+]?\d*\.\d+|\d+", last_line)
        if numbers:
            return numbers[-1]
        return last_line
    return ""

def parse_model_output(output_text):
    """
    Parses the generated response to separate thinking process and the final answer.
    """
    thought = ""
    answer_raw = output_text
    
    if "<think>" in output_text:
        parts = output_text.split("<think>", 1)
        if "</think>" in parts[1]:
            thought_parts = parts[1].split("</think>", 1)
            thought = thought_parts[0].strip()
            answer_raw = thought_parts[1].strip()
        else:
            thought = parts[1].strip()
            answer_raw = ""
            
    extracted_answer = extract_boxed_content(answer_raw)
    return thought, answer_raw, extracted_answer

# Load dataset
import os
data_path = "DATA/train.csv" if os.path.exists("DATA/train.csv") else "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path)

# Run diagnostic evaluation on a subset (e.g. 30 samples)
DIAGNOSTIC_SAMPLES = min(30, len(df))
df_sample = df.head(DIAGNOSTIC_SAMPLES).copy()

print(f"Preparing prompts for {DIAGNOSTIC_SAMPLES} examples using SYSTEM_PROMPT...")
tokenizer = llm.get_tokenizer()
prompts = []
for idx, row in df_sample.iterrows():
    messages = build_prompt(row["prompt"])
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(input_text)

# Sampling parameters matching evaluation environment
sampling_params = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    max_tokens=2048
)

print("Running batch generation via vLLM...")
outputs = llm.generate(prompts, sampling_params)

print("Batch inference complete! Parsing results...")
    results = []
    has_answers = "answer" in df_sample.columns
    for idx, output in enumerate(outputs):
        row = df_sample.iloc[idx]
        generated_text = output.outputs[0].text
        thought, raw_response, final_answer = parse_model_output(generated_text)
        
        is_correct = None
        ground_truth = None
        if has_answers:
            ground_truth = row["answer"]
            is_correct = (final_answer.strip() == str(ground_truth).strip())
        
        results.append({
            "id": row["id"],
            "prompt": row["prompt"],
            "ground_truth": ground_truth,
            "model_thought": thought,
            "model_raw_response": raw_response,
            "model_answer": final_answer,
            "is_correct": is_correct
        })

    df_results = pd.DataFrame(results)
    if has_answers:
        accuracy = df_results["is_correct"].mean()
        print(f"\nDiagnostic Accuracy on {DIAGNOSTIC_SAMPLES} samples: {accuracy * 100:.2f}%")
    else:
        print(f"\nInference completed on {DIAGNOSTIC_SAMPLES} samples (No ground truth 'answer' column found for evaluation).")

    output_csv = "diagnostic_results_simple.csv"
    df_results.to_csv(output_csv, index=False)
    print(f"Diagnostic results saved to: {output_csv}")

    if has_answers:
        # View failure cases
        df_failures = df_results[~df_results["is_correct"]]
        print(f"Total failure cases: {len(df_failures)}\n")
        for i, (_, row) in enumerate(df_failures.head(5).iterrows()):
            print(f"❌ FAILURE CASE {i+1} (ID: {row['id']})")
            print(f"Prompt: {row['prompt']}")
            print(f"Ground Truth: {row['ground_truth']}")
            print(f"Model Extracted Answer: {row['model_answer']}")
            print(f"Model Raw Response: {row['model_raw_response']}")
            print(f"Model Thought Process:\n{row['model_thought']}")
            print("-"*80)
    else:
        # Print the first few model outputs for visual inspection
        print("\nPreview of model predictions:")
        for i, (_, row) in enumerate(df_results.head(5).iterrows()):
            print(f"✨ SAMPLE {i+1} (ID: {row['id']})")
            print(f"Prompt: {row['prompt']}")
            print(f"Model Extracted Answer: {row['model_answer']}")
            print(f"Model Thought Process:\n{row['model_thought']}")
            print("-"*80)